In [2]:
import ollama
import chromadb
import pymupdf4llm
import pathlib
import markdown
from langchain_text_splitters import RecursiveCharacterTextSplitter



In [14]:
md_text = pymupdf4llm.to_markdown("cookies.pdf")
pathlib.Path("output.md").write_bytes(md_text.encode())

4707

In [22]:
client = chromadb.Client()
collection = client.create_collection(name="cookies") # create collection called docs to store embeddings

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)

In [23]:
with open('output.md', 'r', encoding='utf-8') as f:
    file = f.read()

texts = splitter.create_documents([file])
print(f"Number of chunks: {len(texts)}")
for i in range(5):
    print(texts[i].page_content[:250])


Number of chunks: 6
# Cleo Coyle Bakes up an Urban Legend: The Neiman Marcus $250 Cookie Recipe
## _T his is one urban legend that’s easy to swallow! _

_To read my full blog post about this recipe and learn_
_[why this recipe is labeled as worth $250, click here.](http
**Yield:** 2 dozen cookies

**Ingredients:**

1/2 cup (1 stick) butter, softened
1 cup light brown sugar
3 Tablespoons granulated sugar
1 large egg
2 teaspoons vanilla extract
1-3/4 cups all-purpose flour
1/2 teaspoon baking powder
1/2 teaspoon bakin
3. In a mixing bowl, sift together the dry ingredients and beat into
the butter mixture at low speed for about 15 seconds. Stir in the
instant espresso powder and chocolate chips.



4. Using a 1-ounce scoop or a 2-tablespoon measure, drop cookie dou
**CLEO’S TIPS for the best results with**
**this recipe:**

**1 – Hydrate the dough:** For the best results here, I strongly
suggest that you chill this dough in the fridge overnight or 24
to 36 hours before baking. Simply form

In [24]:
# store each document in a vector embedding database
response = ollama.embed(model="nomic-embed-text", input=[str(d) for d in texts]) # turn document (chunk) into vector embedding
collection.add(
  ids=[str(i) for i in range(len(texts))],
  embeddings=response['embeddings'],
  documents=[str(d) for d in texts]
)  # add each embedding to chromadb

In [25]:
# example input
prompt = "What quantity of each ingredient do I need for cookies?"

# generate an embedding for the input and retrieve the most relevant doc
response = ollama.embed(
  model="nomic-embed-text",
  input=prompt
)
results = collection.query(
  query_embeddings=[response["embeddings"][0]],
  n_results=3
) # query db for most relevant chunks

print(len(results['documents']))
# for i in range(3):
#     print(f"Chunk:\n\n\n\n {results['documents'][i]}")
data = results['documents']

1


In [26]:
print(data)

[["page_content='**Yield:** 2 dozen cookies\n\n**Ingredients:**\n\n1/2 cup (1 stick) butter, softened\n1 cup light brown sugar\n3 Tablespoons granulated sugar\n1 large egg\n2 teaspoons vanilla extract\n1-3/4 cups all-purpose flour\n1/2 teaspoon baking powder\n1/2 teaspoon baking soda\n1/2 teaspoon salt\n1-1/2 teaspoons instant espresso powder\n1-1/2 cups semi-sweet chocolate chips\n\n_See my tips on page 2 for getting the best_\n_results out of this recipe. ~ Cleo_\n\n1. Preheat oven to 300° F. Cream the butter with the sugars using\nan electric mixer on medium speed until fluffy (about 30 seconds).\n\n2. Beat in the egg and the vanilla extract for another 30 seconds.\n\n3. In a mixing bowl, sift together the dry ingredients and beat into\nthe butter mixture at low speed for about 15 seconds. Stir in the\ninstant espresso powder and chocolate chips.'", "page_content='in diameter.\n\n**3 – Baking the cookies:** I use a simple $5.00 oven thermometer\nto make sure my oven temperature is a

In [27]:
# generate a response combining the prompt and data we retrieved in step 2
output = ollama.generate(
  model="llama3.2",
  prompt=f"Using this data: {data}. Respond to this prompt: {prompt}."
)

print(output['response'])

According to the recipe, you will need:

* 1/2 cup (1 stick) butter
* 1 cup light brown sugar
* 3 Tablespoons granulated sugar
* 1 large egg
* 2 teaspoons vanilla extract
* 1-3/4 cups all-purpose flour
* 1/2 teaspoon baking powder
* 1/2 teaspoon baking soda
* 1/2 teaspoon salt
* 1-1/2 teaspoons instant espresso powder
* 1-1/2 cups semi-sweet chocolate chips

Note that the quantity of each ingredient is listed in the recipe, but it's worth noting that "1-3/4 cups" and "1-1/2 cups" are ranges rather than exact quantities. If you want to be more precise, you can use a measuring cup or a digital kitchen scale to measure out the ingredients exactly.


In [28]:
output = ollama.generate(
  model="llama3.2",
  prompt=f"What is 2 + 3?"
)

print(output['response'])

The answer to 2 + 3 is 5.


In [10]:
# generate a response combining the prompt and data we retrieved in step 2
output = ollama.generate(
  model="llama3.2",
  prompt=f"Respond to this prompt: {prompt}."
)

print(output['response'])

I'm happy to provide a summary based on publicly available information. However, please note that I don't have direct access to Alphabet's internal documents or the actual report.

That being said, Alphabet's 2022 annual report (Form 10-K) provides an overview of the company's financial performance and key accomplishments for the fiscal year ended December 31, 2022. Here are some highlights:

**Financial Highlights:**

* Net income: $40.8 billion, representing a 15% increase from 2021.
* Revenue: $232.9 billion, up 14% from the previous year.
* Gross cash and short-term investments: $117.4 billion, up from $96.2 billion in 2021.

**Organizational and Business Highlights:**

* Alphabet's subsidiary companies continued to drive growth across various segments:
	+ Google Cloud (GCP) expanded its leadership position, with net sales of $22.7 billion.
	+ YouTube reached 2 billion active monthly users, marking a significant milestone for the platform.
	+ Search advertising revenue continued to

In [ ]:
# generate a response combining the prompt and data we retrieved in step 2
output = ollama.generate(
  model="llama3.2",
  prompt=f"What is this data talking about?: {data}."
)

print(output['response'])